# ytfakenews: Colab quickstart

Transcribe a YouTube video and classify the transcript as REAL or FAKE news with
[ytfakenews](https://github.com/jumincho/yt-fakenews-classifier).

> The classifier learned the style of 2016-era US political news articles. It reacts
> to style and source cues and does not check facts. Read the model card in the
> README before you interpret a result.

The baseline and transcription work on the default CPU runtime. Fine-tuning the
transformer needs a GPU: *Runtime > Change runtime type > T4 GPU*.

## 1. Install

Clone the repository (it contains the training data) and install the package with both optional extras.

In [ ]:
!git clone --depth 1 https://github.com/jumincho/yt-fakenews-classifier.git
%cd yt-fakenews-classifier
!pip install --quiet ".[asr,transformer]"

## 2. Train the baseline

TF-IDF + logistic regression on the bundled dataset (stratified 80/10/10 split, seed 42). This takes under a minute.

In [ ]:
!ytfakenews train baseline

## 3. Optional: fine-tune XLM-RoBERTa

Fine-tunes `xlm-roberta-base` with head+tail truncation and early stopping on
validation F1, then saves it to `models/transformer`. It needs a GPU runtime and
takes considerably longer than the baseline. Set `FINE_TUNE` to try it.

In [ ]:
import torch
from IPython import get_ipython

FINE_TUNE = False  # @param {type:"boolean"}

if FINE_TUNE and torch.cuda.is_available():
    get_ipython().system("ytfakenews train transformer")
elif FINE_TUNE:
    print("No GPU found: switch to a GPU runtime to fine-tune the transformer.")

## 4. Classify a video

Paste a YouTube URL. `ytfakenews run` downloads the audio, transcribes it with
faster-whisper, splits the transcript into chunks and averages P(fake) over them.
Tick `USE_CAPTIONS` to use the video's own subtitles instead of Whisper. Without a
URL the cell classifies the two fictional example transcripts that ship with the
repository.

In [ ]:
from IPython import get_ipython

URL = ""  # @param {type:"string"}
MODEL = "models/baseline"  # @param ["models/baseline", "models/transformer"]
USE_CAPTIONS = False  # @param {type:"boolean"}

shell = get_ipython()
if URL:
    captions = "--captions" if USE_CAPTIONS else ""
    shell.system(f'ytfakenews run "{URL}" --model {MODEL} {captions}')
else:
    for example in ["transcript_local_news.txt", "transcript_sensational.txt"]:
        print(f"examples/{example}")
        shell.system(f"ytfakenews predict examples/{example} --model {MODEL}")
        print()

## 5. Inspect the chunk scores in Python

The same pipeline is available as a Python API. This cell scores the newest
transcript in `outputs/` (or an example) in 150-word chunks.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import pandas as pd

from ytfakenews import classify_text, load_classifier
from ytfakenews.text import read_transcript

transcripts = sorted(Path("outputs").glob("*.txt"), key=lambda path: path.stat().st_mtime)
path = transcripts[-1] if transcripts else Path("examples/transcript_sensational.txt")

prediction = classify_text(
    read_transcript(path), load_classifier(MODEL), chunk_words=150, overlap=30
)
print(f"{path}: {prediction.label} (P(fake) = {prediction.p_fake:.3f})")
pd.DataFrame([asdict(chunk) for chunk in prediction.chunks])